# AIS x Catch Exploratory Analysis

**Question**: does the backfilled Marine Cadastre AIS data (3.9M positions, 2,572 paired boat-days across 2020-2025 Mar-Oct) carry visible signal about catch outcomes for the 14 tracked SD sportfishing boats?

If yes, we invest in a feature-extraction pipeline and integrate into the existing forecaster in `src/forecast.py`. If no, we go a different direction.

Approach in this notebook:
1. Compute per-boat-day AIS stats: distance from port, loiter fraction (sog < 3 kt while at sea), number of distinct 'spots' (1 km grid cells with loiter points).
2. Merge with catch counts by species from `paired_days.csv`.
3. Look at rank correlations between AIS features and catch magnitude.
4. Plot two example days for the same boat  a high-bluefin day and a zero-bluefin day  to see if the trajectory pattern differs visually.
5. Fleet-level co-location signal on top-catch days.

In [1]:
import json
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path(r'C:/Users/Jenelle/Projects/sd-sport-fishing')
DB = ROOT / 'tracker.db'
PAIRED_CSV = ROOT / 'forecast' / 'paired_days.csv'
PLOTS = ROOT / 'forecast' / 'plots'
PLOTS.mkdir(exist_ok=True)

PORT_LAT, PORT_LON = 32.71, -117.22  # SD bay landings (H&M / Fisherman's)
LOITER_KT = 3.0
AT_SEA_KM = 5.0

# --- Load paired boat-days + parse species JSON.
paired = pd.read_csv(PAIRED_CSV)

def parse_species(s):
    try:
        d = json.loads(s)
    except Exception:
        return {}
    out = {}
    for k, v in d.items():
        if isinstance(v, (int, float)):
            out[k.lower()] = out.get(k.lower(), 0) + int(v)
    return out

def bluefin_total(species):
    return sum(v for k, v in species.items() if 'bluefin' in k)

def sum_species(species, key):
    return sum(v for k, v in species.items() if key in k)

paired['species']       = paired['catch_species_json'].apply(parse_species)
paired['catch_total']   = paired['species'].apply(lambda d: sum(d.values()))
paired['catch_bluefin'] = paired['species'].apply(bluefin_total)
paired['catch_yft']     = paired['species'].apply(lambda d: sum_species(d, 'yellowfin'))
paired['catch_ytail']   = paired['species'].apply(lambda d: sum_species(d, 'yellowtail'))

print(f'paired boat-days:    {len(paired):,}')
print(f'total catch (sum):   {paired["catch_total"].sum():,}')
print(f'total bluefin:       {paired["catch_bluefin"].sum():,}')
print(f'days w/ bluefin > 0: {(paired["catch_bluefin"] > 0).sum():,}')
print(f'distinct boats:      {paired["boat"].nunique()}')
paired[['boat','mmsi','date','n_positions','catch_total','catch_bluefin']].head()

paired boat-days:    2,572
total catch (sum):   168,998
total bluefin:       43,749
days w/ bluefin > 0: 1,522
distinct boats:      11


,boat,mmsi,date,n_positions,catch_total,catch_bluefin
0,Liberty,338301392,2020-06-26,14,86,0
1,Liberty,338301392,2020-07-05,7,182,0
2,Liberty,338301392,2020-07-16,23,78,61
3,Liberty,338301392,2020-07-17,37,10,9
4,Liberty,338301392,2020-07-26,56,4,4


In [2]:
# --- Load ALL positions once, bucket to Pacific date, compute per-day AIS stats.
print('loading positions ...')
con = sqlite3.connect(DB)
pos = pd.read_sql(
    "SELECT mmsi, timestamp, lat, lon, sog FROM positions "
    "WHERE source='marine_cadastre'", con)
con.close()
print(f'  loaded {len(pos):,} rows')

# Timestamps come in two shapes ('2020-03-01T00:00:01Z' and '2025-08-01 00:00:01Z').
# format='ISO8601' handles both; utc=True keeps naive parses in UTC.
pos['ts'] = pd.to_datetime(pos['timestamp'], format='ISO8601', utc=True)
pos['pac_date'] = pos['ts'].dt.tz_convert('America/Los_Angeles').dt.strftime('%Y-%m-%d')

# Haversine distance from SD port for every position (vectorised).
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1 = np.radians(lat1), np.radians(lon1)
    lat2, lon2 = np.radians(lat2), np.radians(lon2)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

pos['dist_km']  = haversine_km(PORT_LAT, PORT_LON, pos['lat'].values, pos['lon'].values)
pos['at_sea']   = pos['dist_km'] > AT_SEA_KM
pos['loiter']   = pos['at_sea'] & (pos['sog'].fillna(999) < LOITER_KT)
pos['grid_key'] = (pos['lat'].round(2).astype(str) + '_' + pos['lon'].round(2).astype(str))

print('computing per-boat-day stats ...')
grp = pos.groupby(['mmsi', 'pac_date'])
stats = grp.agg(
    n_positions       = ('sog',    'count'),
    at_sea_positions  = ('at_sea', 'sum'),
    loiter_positions  = ('loiter', 'sum'),
    max_dist_km       = ('dist_km','max'),
    mean_at_sea_dist  = ('dist_km', 'mean'),
    max_sog           = ('sog',    'max'),
).reset_index()

# Number of unique 1-km grid cells with loiter points  proxy for 'distinct spots tried'.
spots = pos[pos['loiter']].groupby(['mmsi','pac_date'])['grid_key'].nunique().rename('loiter_spots').reset_index()
stats = stats.merge(spots, on=['mmsi','pac_date'], how='left').fillna({'loiter_spots': 0})
stats['loiter_frac'] = np.where(stats['at_sea_positions'] > 0,
                                stats['loiter_positions'] / stats['at_sea_positions'], 0.0)

# 'Left port' filter: only count days where the boat exceeded 5 kt at some point,
# so days where the boat never moved (transponder in port) don't skew things.
stats['went_fishing'] = stats['max_sog'] >= 5.0
print(f'  {len(stats):,} distinct (mmsi, Pacific-date) buckets')
print(f'  {stats["went_fishing"].sum():,} of them look like actual fishing days')
stats.head()

loading positions ...


  loaded 3,927,339 rows


computing per-boat-day stats ...


  8,825 distinct (mmsi, Pacific-date) buckets
  4,394 of them look like actual fishing days


,mmsi,pac_date,n_positions,at_sea_positions,loiter_positions,max_dist_km,mean_at_sea_dist,max_sog,loiter_spots,loiter_frac,went_fishing
0,338301392,2020-03-07,43,0,0,2.410708,1.694221,14.4,0.0,0.0,True
1,338301392,2020-03-14,20,0,0,1.242842,1.242326,0.0,0.0,0.0,False
2,338301392,2020-04-10,1,0,0,1.239946,1.239946,0.0,0.0,0.0,False
3,338301392,2020-05-04,10,0,0,1.244467,1.243491,0.0,0.0,0.0,False
4,338301392,2020-05-06,11,0,0,1.243383,1.241426,0.0,0.0,0.0,False


In [3]:
# --- Merge AIS stats with paired-days catch data.
paired['mmsi_str']    = paired['mmsi'].astype(str)
stats['mmsi_str']     = stats['mmsi'].astype(str)
df = paired.merge(
    stats[['mmsi_str','pac_date','n_positions','loiter_frac','loiter_spots',
           'max_dist_km','mean_at_sea_dist','max_sog','went_fishing']]
        .rename(columns={'pac_date':'date','n_positions':'n_pos_ais'}),
    on=['mmsi_str','date'], how='left'
)
print('merged rows:', len(df))
print('rows with AIS stats:', df['loiter_frac'].notna().sum())
print()
print('Distribution of paired days by year:')
print(df['date'].str[:4].value_counts().sort_index())
print()
print('Paired days per tracked boat:')
print(df.groupby('boat').size().sort_values(ascending=False))

merged rows: 2572
rows with AIS stats: 2572

Distribution of paired days by year:
date
2020    450
2021    500
2022    501
2023    386
2024    371
2025    364
Name: count, dtype: int64

Paired days per tracked boat:
boat
Grande                 741
Liberty                640
Legend                 408
Voyager                396
Fortune                166
Pacific Queen           87
Constitution            58
Spirit of Adventure     29
Islander                23
Polaris Supreme         18
Royal Polaris            6
dtype: int64


In [4]:
# --- Rank correlations between AIS features and catch outcomes.
sub = df.dropna(subset=['loiter_frac']).copy()
sub = sub[sub['went_fishing']]  # exclude days where the boat clearly didn't leave port

features = ['loiter_frac', 'loiter_spots', 'max_dist_km', 'mean_at_sea_dist',
            'max_sog', 'n_pos_ais']
targets  = ['catch_total', 'catch_bluefin', 'catch_yft', 'catch_ytail']

corr = pd.DataFrame(index=features, columns=targets, dtype=float)
for f in features:
    for t in targets:
        if sub[t].sum() == 0:
            corr.loc[f, t] = np.nan
            continue
        corr.loc[f, t] = sub[[f, t]].corr(method='spearman').iloc[0, 1]

print(f'Spearman rank correlations (n={len(sub):,} boat-days after went_fishing filter):')
print(corr.round(3))
print()

# How many of these boat-days had zero catch (skunked)?
skunk_share = (sub['catch_total'] == 0).mean()
print(f'skunked share (catch_total==0): {skunk_share:.1%}')

# Quick check on separation: mean AIS feature values on high-bluefin vs. no-bluefin days.
bf_top = sub[sub['catch_bluefin'] >= sub['catch_bluefin'].quantile(0.9)]
bf_none = sub[sub['catch_bluefin'] == 0]
print()
print(f'top-decile-bluefin days (n={len(bf_top)}) vs zero-bluefin days (n={len(bf_none)}):')
for f in features:
    a, b = bf_top[f].mean(), bf_none[f].mean()
    print(f'  {f:<20} top={a:>10.2f}   zero={b:>10.2f}   ratio={a/b if b else float("nan"):.2f}')

Spearman rank correlations (n=1,938 boat-days after went_fishing filter):
                  catch_total  catch_bluefin  catch_yft  catch_ytail
loiter_frac             0.140          0.007      0.026        0.047
loiter_spots            0.122          0.016      0.026        0.030
max_dist_km             0.056          0.232      0.023       -0.051
mean_at_sea_dist        0.073          0.156      0.037       -0.039
max_sog                -0.123         -0.095     -0.061        0.014
n_pos_ais               0.058          0.166      0.000       -0.024

skunked share (catch_total==0): 3.7%

top-decile-bluefin days (n=197) vs zero-bluefin days (n=837):
  loiter_frac          top=      0.09   zero=      0.07   ratio=1.28
  loiter_spots         top=      3.22   zero=      1.94   ratio=1.66
  max_dist_km          top=     91.26   zero=     76.05   ratio=1.20
  mean_at_sea_dist     top=     39.06   zero=     46.82   ratio=0.83
  max_sog              top=     11.33   zero=     11.41   ratio=0.

In [5]:
# --- Pick a high-bluefin and zero-bluefin day for the SAME boat, plot trajectories.
def plot_day(mmsi_str, date_str, title, ax):
    mm = int(mmsi_str)
    day_pts = pos[(pos['mmsi'] == mm) & (pos['pac_date'] == date_str)].copy()
    day_pts = day_pts.sort_values('ts')
    if day_pts.empty:
        ax.set_title(f'{title}\n(no positions)')
        return
    at_sea = day_pts[day_pts['at_sea']]
    loit   = day_pts[day_pts['loiter']]
    ax.scatter(day_pts['lon'], day_pts['lat'], c=day_pts['sog'].clip(0, 15),
               cmap='viridis', s=6, alpha=0.6)
    ax.scatter(loit['lon'], loit['lat'], facecolors='none', edgecolors='red',
               s=40, linewidths=1.2, label=f'loiter (sog<{LOITER_KT}kt)')
    # port marker
    ax.plot(PORT_LON, PORT_LAT, marker='^', color='black', markersize=10)
    ax.annotate('SD port', (PORT_LON, PORT_LAT), textcoords='offset points',
                xytext=(6, 6), fontsize=9)
    ax.set_xlabel('lon')
    ax.set_ylabel('lat')
    ax.set_title(title)
    ax.legend(loc='lower left', fontsize=8)
    ax.grid(alpha=0.3)

# Find a boat with both a top-bluefin and a zero-bluefin day, both with AIS coverage.
candidates = sub.copy()
boat_pool = (candidates.groupby('boat')
             .filter(lambda g: (g['catch_bluefin'] >= 20).any() and (g['catch_bluefin'] == 0).any())
             .groupby('boat').size().sort_values(ascending=False))
print('Boats with both high-bluefin and zero-bluefin paired days:')
print(boat_pool.head())

chosen_boat = boat_pool.index[0]
boat_days = candidates[candidates['boat'] == chosen_boat]
hi = boat_days.sort_values('catch_bluefin', ascending=False).iloc[0]
lo = boat_days[boat_days['catch_bluefin'] == 0].sort_values('n_pos_ais', ascending=False).iloc[0]

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
plot_day(str(hi['mmsi']), hi['date'],
         f'{chosen_boat} {hi["date"]}  bluefin={int(hi["catch_bluefin"])}  total={int(hi["catch_total"])}',
         axes[0])
plot_day(str(lo['mmsi']), lo['date'],
         f'{chosen_boat} {lo["date"]}  bluefin=0  total={int(lo["catch_total"])}',
         axes[1])
fig.suptitle(f'Trajectory comparison  same boat, high-bluefin vs zero-bluefin', fontsize=13)
fig.tight_layout()
out = PLOTS / '01_trajectory_comparison.png'
fig.savefig(out, dpi=120, bbox_inches='tight')
print(f'saved {out.relative_to(ROOT)}')
plt.close(fig)

Boats with both high-bluefin and zero-bluefin paired days:
boat
Grande          741
Liberty         529
Legend          399
Constitution     56
Voyager          42
dtype: int64


saved forecast\plots\01_trajectory_comparison.png


In [6]:
# --- Scatter: loiter_frac and max_dist_km vs. bluefin catch.
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

s = sub[sub['catch_bluefin'] >= 0]
axes[0].scatter(s['loiter_frac'], s['catch_bluefin'], alpha=0.35, s=14)
axes[0].set_xlabel('loiter fraction (sog<3kt while at sea)')
axes[0].set_ylabel('bluefin catch')
axes[0].set_title(f'loiter_frac vs bluefin  spearman={corr.loc["loiter_frac","catch_bluefin"]:.3f}')
axes[0].grid(alpha=0.3)

axes[1].scatter(s['max_dist_km'], s['catch_bluefin'], alpha=0.35, s=14, color='C1')
axes[1].set_xlabel('max distance from port (km)')
axes[1].set_ylabel('bluefin catch')
axes[1].set_title(f'max_dist_km vs bluefin  spearman={corr.loc["max_dist_km","catch_bluefin"]:.3f}')
axes[1].grid(alpha=0.3)

fig.tight_layout()
out = PLOTS / '02_correlations.png'
fig.savefig(out, dpi=120, bbox_inches='tight')
print(f'saved {out.relative_to(ROOT)}')
plt.close(fig)

saved forecast\plots\02_correlations.png


In [7]:
# --- Fleet co-location: on days when multiple tracked boats broadcast,
# do their loiter clusters overlap on high-catch days more than on low-catch days?

# Total bluefin caught by the tracked fleet per Pacific date.
fleet_by_day = paired.groupby('date').agg(
    fleet_bluefin  = ('catch_bluefin', 'sum'),
    fleet_total    = ('catch_total',   'sum'),
    n_boats_paired = ('boat',          'nunique'),
).reset_index()

# Grid cells with loiter across the fleet per day.
loiter_grid = pos[pos['loiter']].groupby(['pac_date','grid_key'])['mmsi'].nunique().reset_index(name='n_boats_in_cell')
hot_grid = loiter_grid[loiter_grid['n_boats_in_cell'] >= 2]

colo = hot_grid.groupby('pac_date').agg(
    n_shared_cells    = ('grid_key', 'nunique'),
    max_boats_in_cell = ('n_boats_in_cell', 'max'),
).reset_index().rename(columns={'pac_date':'date'})

fleet_by_day = fleet_by_day.merge(colo, on='date', how='left').fillna(
    {'n_shared_cells': 0, 'max_boats_in_cell': 0})

print(f'fleet-day rows: {len(fleet_by_day):,}')
print(f'days with >=2-boat shared loiter cells: {(fleet_by_day["n_shared_cells"] > 0).sum():,}')
print()

# On days where the fleet caught many bluefin, do they cluster more?
top_days = fleet_by_day[fleet_by_day['fleet_bluefin'] >= fleet_by_day['fleet_bluefin'].quantile(0.9)]
rest_days = fleet_by_day[fleet_by_day['fleet_bluefin'] < fleet_by_day['fleet_bluefin'].quantile(0.9)]
print(f'top-decile fleet-bluefin days: n={len(top_days)}')
print(f'  mean shared cells: {top_days["n_shared_cells"].mean():.2f}')
print(f'  mean max_boats_in_cell: {top_days["max_boats_in_cell"].mean():.2f}')
print(f'rest of days:                  n={len(rest_days)}')
print(f'  mean shared cells: {rest_days["n_shared_cells"].mean():.2f}')
print(f'  mean max_boats_in_cell: {rest_days["max_boats_in_cell"].mean():.2f}')

# Rank correlation, controlling for n_boats_paired.
s = fleet_by_day[fleet_by_day['n_boats_paired'] >= 2]
print()
print(f'On days with >=2 paired boats (n={len(s):,}):')
print(f'  spearman(fleet_bluefin, n_shared_cells)    = '
      f'{s[["fleet_bluefin","n_shared_cells"]].corr(method="spearman").iloc[0,1]:.3f}')
print(f'  spearman(fleet_bluefin, max_boats_in_cell) = '
      f'{s[["fleet_bluefin","max_boats_in_cell"]].corr(method="spearman").iloc[0,1]:.3f}')
print(f'  spearman(fleet_total,   n_shared_cells)    = '
      f'{s[["fleet_total","n_shared_cells"]].corr(method="spearman").iloc[0,1]:.3f}')

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(fleet_by_day['n_shared_cells'], fleet_by_day['fleet_bluefin'],
           alpha=0.4, s=18)
ax.set_xlabel('# shared loiter cells (>=2 tracked boats)')
ax.set_ylabel('fleet bluefin catch that day')
ax.set_title('Fleet co-location vs. fleet bluefin catch')
ax.grid(alpha=0.3)
fig.tight_layout()
out = PLOTS / '03_fleet_colocation.png'
fig.savefig(out, dpi=120, bbox_inches='tight')
print(f'\nsaved {out.relative_to(ROOT)}')
plt.close(fig)

fleet-day rows: 1,031
days with >=2-boat shared loiter cells: 149

top-decile fleet-bluefin days: n=108
  mean shared cells: 0.74
  mean max_boats_in_cell: 0.36
rest of days:                  n=923
  mean shared cells: 0.54
  mean max_boats_in_cell: 0.29

On days with >=2 paired boats (n=751):
  spearman(fleet_bluefin, n_shared_cells)    = 0.054
  spearman(fleet_bluefin, max_boats_in_cell) = 0.049
  spearman(fleet_total,   n_shared_cells)    = -0.050

saved forecast\plots\03_fleet_colocation.png


## Findings

(Filled in by the actual run outputs above  see report.)